# CardioCare — EDA and Preprocessing

**Ethics:** This system informs cardiologists; it does not make autonomous diagnoses.

This notebook explores the UCI Cleveland heart disease dataset and documents how EDA findings shaped the preprocessing pipeline in `src/preprocessing.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.constants import FEATURE_COLUMNS, TARGET_COLUMN
from src.preprocessing import binarize_target, load_raw_dataframe

sns.set_theme(style="whitegrid")

In [ ]:
data_path = PROJECT_ROOT / "data" / "heart_disease.csv"
if not data_path.exists():
    from data.download_data import download_heart_disease
    download_heart_disease(data_path)

raw_df = load_raw_dataframe(data_path)
df = binarize_target(raw_df)
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
target_distribution = df[TARGET_COLUMN].value_counts(normalize=True)
print(target_distribution)

target_distribution.plot(kind="bar", title="Target Class Distribution", color=["#4C78A8", "#F58518"])
plt.ylabel("Proportion")
plt.xlabel("Class (0=healthy, 1=disease)")
plt.show()

The dataset is moderately imbalanced. Accuracy alone can hide poor sensitivity, so we prioritize **recall** and **balanced accuracy** during model selection.

In [ ]:
missing_by_column = raw_df.isna().mean().sort_values(ascending=False)
print("Missing rate by column:")
print(missing_by_column)
print(f"\nTotal missing cells: {raw_df.isna().sum().sum()}")

In [ ]:
continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(1, len(continuous_features), figsize=(16, 4))
for axis, feature in zip(axes, continuous_features):
    sns.boxplot(y=df[feature], ax=axis, color="#72B7B2")
    axis.set_title(feature)
plt.suptitle("Outlier Inspection for Continuous Features")
plt.tight_layout()
plt.show()

In [ ]:
duplicate_count = df.duplicated().sum()
empty_columns = [col for col in df.columns if df[col].isna().all()]
print(f"Duplicate rows: {duplicate_count}")
print(f"Empty columns: {empty_columns}")

In [ ]:
from sklearn.model_selection import train_test_split
from src.constants import RANDOM_SEED
from src.preprocessing import build_preprocessing_pipeline, split_features_target

features, target = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=target,
)

preprocess_pipeline = build_preprocessing_pipeline()
preprocess_pipeline.fit(X_train)
X_train_processed = preprocess_pipeline.transform(X_train)
X_test_processed = preprocess_pipeline.transform(X_test)
print("Train processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)

## EDA → Preprocessing Summary

EDA showed:
1. Missing values encoded as `?`, especially in `ca` and `thal`.
2. Mild class imbalance that makes recall more important than raw accuracy.
3. Visible outliers in `chol`, `trestbps`, and `oldpeak`.
4. A small number of duplicate rows and no completely empty columns.

Therefore preprocessing:
- Replaces `?` with `np.nan` and uses `pd.isna()` for detection.
- Drops columns only when missing rate exceeds 30%.
- Imputes numeric features with the **median** and categorical features with the **mode**.
- Clips outliers using train-only IQR bounds.
- Fits exclusively on the training split to prevent data leakage.